# Day 5 - Large Language Model (LLM) Fundamentals

## Task

This notebook demonstrates the fundamental concepts required for modern Large Language Models.

Topics Covered

- Tokenization
- BPE
- WordPiece
- SentencePiece
- Embeddings
- Positional Embeddings
- Attention Mask
- Hugging Face Tokenizers

Author:
Hassan Iqbal

Internship:
Generative AI Internship
National Centre of Artificial Intelligence (NCAI)

In [1]:
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModel
)

import torch

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


# What is Tokenization?

Tokenization is the process of converting raw text into smaller units called tokens.

These tokens are later converted into numerical IDs before being processed by a Transformer model.

In [2]:
sentence = "I love studying Large Language Models."

print(sentence)

I love studying Large Language Models.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer Loaded")

/mnt/dataD/NCAI Internship/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer Loaded


In [4]:
tokens = tokenizer.tokenize(sentence)

print(tokens)

['i', 'love', 'studying', 'large', 'language', 'models', '.']


In [5]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[1045, 2293, 5702, 2312, 2653, 4275, 1012]


In [6]:
encoded = tokenizer(sentence)

encoded

{'input_ids': [101, 1045, 2293, 5702, 2312, 2653, 4275, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
decoded = tokenizer.decode(
    encoded["input_ids"]
)

print(decoded)

I0000 00:00:1783068515.058063   26586 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[CLS] i love studying large language models. [SEP]


# Special Tokens

Transformers use special tokens such as

CLS

SEP

PAD

MASK

Each token has a specific purpose.

In [8]:
print(tokenizer.special_tokens_map)

{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


# WordPiece Tokenization

BERT uses WordPiece tokenization.

Example

playing

↓

play

##ing

In [9]:
words = [
    "playing",
    "unbelievable",
    "internationalization"
]

for w in words:

    print(w)

    print(tokenizer.tokenize(w))

    print()

playing
['playing']

unbelievable
['unbelievable']

internationalization
['international', '##ization']



# BPE Tokenization

GPT models use Byte Pair Encoding (BPE).

Unlike WordPiece, BPE merges the most frequent character pairs.

In [10]:
gpt = AutoTokenizer.from_pretrained(
    "gpt2"
)

print(gpt.tokenize(sentence))

/mnt/dataD/NCAI Internship/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['I', 'Ġlove', 'Ġstudying', 'ĠLarge', 'ĠLanguage', 'ĠModels', '.']


# SentencePiece

T5 uses SentencePiece tokenization.

SentencePiece works directly on raw text without relying on spaces.

In [11]:
t5 = AutoTokenizer.from_pretrained(
    "t5-small"
)

print(
    t5.tokenize(sentence)
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['▁I', '▁love', '▁studying', '▁Large', '▁Language', '▁Model', 's', '.']


# Comparison of Tokenizers

In [12]:
print("BERT")

print(tokenizer.tokenize(sentence))

print()

print("GPT")

print(gpt.tokenize(sentence))

print()

print("T5")

print(t5.tokenize(sentence))

BERT
['i', 'love', 'studying', 'large', 'language', 'models', '.']

GPT
['I', 'Ġlove', 'Ġstudying', 'ĠLarge', 'ĠLanguage', 'ĠModels', '.']

T5
['▁I', '▁love', '▁studying', '▁Large', '▁Language', '▁Model', 's', '.']


# Embeddings

After tokenization, every token is converted into a dense vector representation called an embedding.

These vectors capture semantic meaning.

In [13]:
model = AutoModel.from_pretrained(
    "bert-base-uncased"
)

print("Model Loaded")

/mnt/dataD/NCAI Internship/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model Loaded


In [14]:
inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

outputs = model(**inputs)

print(
    outputs.last_hidden_state.shape
)

torch.Size([1, 9, 768])


# Positional Embeddings

Transformers process all words simultaneously.

Therefore, positional information is added to every embedding so that the model understands word order.

In [15]:
s1 = "Dog bites man"

s2 = "Man bites dog"

print(tokenizer.tokenize(s1))

print(tokenizer.tokenize(s2))

['dog', 'bites', 'man']
['man', 'bites', 'dog']


# Attention Mask

Attention Mask tells the model which tokens are real and which are padding.

In [16]:
print(inputs["attention_mask"])

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])


# Complete Pipeline

Raw Text

↓

Tokenizer

↓

Token IDs

↓

Attention Mask

↓

Embeddings

↓

Positional Embeddings

↓

Transformer

↓

Prediction

In [17]:
print("="*60)

print("Notebook Completed Successfully")

print("="*60)

Notebook Completed Successfully


In [18]:
print("BERT Vocabulary :", tokenizer.vocab_size)
print("GPT Vocabulary  :", gpt.vocab_size)
print("T5 Vocabulary   :", t5.vocab_size)

BERT Vocabulary : 30522
GPT Vocabulary  : 50257
T5 Vocabulary   : 32100


In [19]:
words = [
    "ChatGPT",
    "internationalization",
    "machinelearning",
    "unbelievable"
]

for w in words:
    print("="*50)
    print(w)
    print("BERT :", tokenizer.tokenize(w))
    print("GPT  :", gpt.tokenize(w))
    print("T5   :", t5.tokenize(w))

ChatGPT
BERT : ['chat', '##gp', '##t']
GPT  : ['Chat', 'G', 'PT']
T5   : ['▁Chat', 'GP', 'T']
internationalization
BERT : ['international', '##ization']
GPT  : ['international', 'ization']
T5   : ['▁international', 'ization']
machinelearning
BERT : ['machine', '##lea', '##rn', '##ing']
GPT  : ['machine', 'learning']
T5   : ['▁machine', 'learning']
unbelievable
BERT : ['unbelievable']
GPT  : ['un', 'bel', 'iev', 'able']
T5   : ['▁unbelievable']


In [20]:
print(model.config.hidden_size)

768


# Conclusion

In this notebook, we explored the fundamental preprocessing steps used by modern Large Language Models.

Topics Covered

- Tokenization
- WordPiece
- BPE
- SentencePiece
- Token IDs
- Attention Mask
- Embeddings
- Positional Embeddings

These concepts form the foundation of Transformer-based architectures such as BERT, GPT, T5, LLaMA, and other modern LLMs.